# NB09B — Reviewer Additional Analyses

This notebook adds **review-driven secondary analyses** without modifying NB01–NB08.

It covers:

1. A fixed, wider **SVR hyperparameter sensitivity analysis** under the original egg-disjoint nested splits.
2. An **operational storage-phase classification** derived from frozen OOF regression predictions:
   - Early: days 0–7
   - Middle: days 8–14
   - Late: days 15–21

   These are storage-time phases, **not biological freshness/safety classes**.
3. Quantification of prediction attenuation using the regression of **predicted storage time on observed storage time** for SVR, PLSR and ANN.
4. Text-ready tables for manuscript/supplementary material.

All new outputs are explicitly secondary and review-driven.

**Optimization note.** Spectral preprocessing and scaling are cached once per inner fold; this reduces runtime without changing the fixed grid, splits, model selection rule, or numerical definition of the analysis.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json, hashlib, time, warnings
import numpy as np
import pandas as pd

ROOT = Path('/content/drive/MyDrive/NIR_HUEVOS_PAPER_REBUILD_2026')
RAW = ROOT / '01_DATA_RAW' / 'dataset_egg_storage_RAW.csv'
SPLITS = ROOT / '03_SPLITS_FROZEN'
NB03_DIR = ROOT / '05_RESULTS' / 'NB03_CHEMOMETRIC_BASELINES'
NB04_DIR = ROOT / '05_RESULTS' / 'NB04_DEEP_LEARNING_BENCHMARK'
NB06_DIR = ROOT / '05_RESULTS' / 'NB06_STATISTICAL_ROBUSTNESS'
OUT_DIR = ROOT / '05_RESULTS' / 'NB09B_REVIEWER_ADDITIONAL_ANALYSES'
OUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_REVISION = 'NB09B_v1_review_secondary_analyses'
EXPECTED_DATASET_SHA256 = 'cd5021c555ae6b57f892549c574599cef75edf87f58b3f7f4d246ade9327d15e'

Mounted at /content/drive


In [3]:
# 1. Audit raw input
def sha256_file(path):
    h = hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1024*1024), b''):
            h.update(b)
    return h.hexdigest()

assert sha256_file(RAW) == EXPECTED_DATASET_SHA256
df = pd.read_csv(RAW)
spec_cols = sorted([c for c in df.columns if c.startswith('Spectra_')],
                   key=lambda x:int(x.split('_')[1]))
X = df[spec_cols].to_numpy(float)
y = df.storage_days.to_numpy(float)
samples = df['sample'].to_numpy()
assert len(spec_cols) == 331 and df.shape == (660,333)
print('PASS — dataset integrity verified.')

PASS — dataset integrity verified.


In [4]:
# 2. Reconstruct exact frozen outer/inner egg assignments
outer_map = pd.read_csv(SPLITS / 'outer_group_assignment_seed2026.csv')

inner_maps = {}
for outer in range(1,6):
    p = SPLITS / f'inner_group_assignment_outer{outer:02d}.csv'
    inner_maps[outer] = pd.read_csv(p)

print(outer_map.head())
print(inner_maps[1].head())

   sample  outer_fold
0       1           3
1       2           5
2       3           4
3       4           2
4       5           3
   sample  inner_fold
0       1           4
1       2           4
2       3           1
3       4           1
4       5           2


In [5]:
# 3. Train-only spectral preprocessing
from scipy.signal import savgol_filter
from sklearn.preprocessing import StandardScaler

class SpectralPreprocessor:
    def __init__(self, method):
        self.method = method
        self.ref = None
        self.scaler = None

    def _base(self, X, fit=False):
        X = np.asarray(X,float)
        if self.method == 'raw':
            return X.copy()
        if self.method == 'snv':
            mu = X.mean(1,keepdims=True); sd = X.std(1,keepdims=True)
            sd[sd==0] = 1
            return (X-mu)/sd
        if self.method == 'msc':
            if fit:
                self.ref = X.mean(0)
            if self.ref is None:
                raise RuntimeError('MSC ref not fitted')
            r = self.ref
            rc = r-r.mean()
            den = np.sum(rc**2)
            xm = X.mean(1,keepdims=True)
            slope = np.sum((X-xm)*rc[None,:],axis=1)/den
            slope[np.abs(slope)<1e-12] = 1
            inter = X.mean(1)-slope*r.mean()
            return (X-inter[:,None])/slope[:,None]
        if self.method == 'sg_smooth':
            return savgol_filter(X,11,2,deriv=0,axis=1,mode='interp')
        if self.method == 'sg_deriv1':
            return savgol_filter(X,11,2,deriv=1,delta=1.0,axis=1,mode='interp')
        raise ValueError(self.method)

    def fit(self,X):
        A = self._base(X,fit=True)
        self.scaler = StandardScaler().fit(A)
        return self

    def transform(self,X):
        return self.scaler.transform(self._base(X,fit=False))

    def fit_transform(self,X):
        return self.fit(X).transform(X)

## A. SVR sensitivity analysis

The primary NB03 grid remains frozen and unchanged. This notebook performs a **review-requested post hoc sensitivity analysis**.

To target the actual boundary issue rather than reopen the entire model-selection exercise:

- spectral preprocessing is fixed to **SG first derivative**, selected in all five primary SVR outer folds;
- only \(C\), \(\epsilon\), and \(\gamma\) are expanded;
- the wider grid is fixed in the next cell **before execution**;
- selection still occurs exclusively inside the original egg-disjoint inner folds;
- outer-test eggs remain untouched until each sensitivity configuration has been selected.

These results are supplementary stability evidence and never replace NB03.

In [6]:
# 4. Fixed wider SVR sensitivity grid — do not edit after running
SVR_SENSITIVITY_GRID = {
    'C': [1e4, 1e5, 3e5, 1e6, 3e6],
    'epsilon': [0.5, 1.0, 2.0, 3.0, 4.0],
    'gamma': [1e-6, 3e-6, 1e-5, 3e-5, 1e-4],
}
PREPROCESSING_FIXED = 'sg_deriv1'
print(SVR_SENSITIVITY_GRID)
(OUT_DIR / 'NB09B_SVR_sensitivity_grid.json').write_text(
    json.dumps({'preprocessing_fixed':PREPROCESSING_FIXED, **SVR_SENSITIVITY_GRID}, indent=2)
)

{'C': [10000.0, 100000.0, 300000.0, 1000000.0, 3000000.0], 'epsilon': [0.5, 1.0, 2.0, 3.0, 4.0], 'gamma': [1e-06, 3e-06, 1e-05, 3e-05, 0.0001]}


261

In [7]:
# 5. Helpers for exact egg-disjoint sensitivity search
# OPTIMIZED: preprocessing/scaling is cached once per train/validation split,
# rather than recomputed for every (C, epsilon, gamma) combination.
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler as SKStandardScaler
from itertools import product

def metrics(y_true,pred):
    e=np.asarray(pred)-np.asarray(y_true); ae=np.abs(e)
    return {
        'MAE_days':float(ae.mean()),
        'RMSE_days':float(np.sqrt(np.mean(e**2))),
        'R2':float(r2_score(y_true,pred)),
        'bias_days':float(e.mean())
    }

def idx_for_eggs(egg_ids):
    return np.flatnonzero(np.isin(samples, list(egg_ids)))

def outer_eggs(fold):
    if {'sample','outer_fold'}.issubset(outer_map.columns):
        test = set(outer_map.loc[outer_map.outer_fold==fold,'sample'].astype(int))
    else:
        raise ValueError(f'Unexpected outer split columns: {outer_map.columns.tolist()}')
    train = set(range(1,31)) - test
    return train,test

def inner_eggs(outer, inner):
    m=inner_maps[outer]
    cols=m.columns.tolist()
    if {'sample','inner_fold'}.issubset(m.columns):
        val=set(m.loc[m.inner_fold==inner,'sample'].astype(int))
    elif {'sample','fold'}.issubset(m.columns):
        val=set(m.loc[m['fold']==inner,'sample'].astype(int))
    else:
        raise ValueError(f'Unexpected inner split columns: {cols}')
    otr,_=outer_eggs(outer)
    train=otr-val
    return train,val

def prepare_svr_matrices(train_idx, test_idx):
    pp=SpectralPreprocessor(PREPROCESSING_FIXED)
    A0=pp.fit_transform(X[train_idx]); B0=pp.transform(X[test_idx])
    sc=SKStandardScaler().fit(A0)
    A=sc.transform(A0); B=sc.transform(B0)
    return A, B

def fit_predict_svr_prepared(A, y_train, B, C, epsilon, gamma):
    m=SVR(kernel='rbf',C=C,epsilon=epsilon,gamma=gamma)
    m.fit(A,y_train)
    return m.predict(B)

In [8]:
# 6. Execute sensitivity search — optimized with cached preprocessing matrices
search_path = OUT_DIR / 'NB09B_SVR_sensitivity_inner_search.csv'
selected_path = OUT_DIR / 'NB09B_SVR_sensitivity_selected_configs.csv'
oof_path = OUT_DIR / 'NB09B_SVR_sensitivity_oof.csv'

all_combos = list(product(
    SVR_SENSITIVITY_GRID['C'],
    SVR_SENSITIVITY_GRID['epsilon'],
    SVR_SENSITIVITY_GRID['gamma']
))
print('Hyperparameter combinations:', len(all_combos))

search_rows=[]
selected_rows=[]
oof_rows=[]

for outer in range(1,6):
    outer_train, outer_test = outer_eggs(outer)

    # Cache each of the four inner-fold preprocessed matrices once.
    inner_cache = {}
    for inner in range(1,5):
        itr,iva=inner_eggs(outer,inner)
        tr=idx_for_eggs(itr); va=idx_for_eggs(iva)
        A,B=prepare_svr_matrices(tr,va)
        inner_cache[inner]=(A, y[tr].copy(), B, y[va].copy())

    combo_scores=[]
    for C,eps,gam in all_combos:
        maes=[]
        for inner in range(1,5):
            A,ytr,B,yva=inner_cache[inner]
            pred=fit_predict_svr_prepared(A,ytr,B,C,eps,gam)
            mae=mean_absolute_error(yva,pred)
            maes.append(mae)
            search_rows.append({
                'outer_fold':outer,'inner_fold':inner,'C':C,'epsilon':eps,'gamma':gam,
                'MAE_days':mae
            })
        combo_scores.append({
            'C':C,'epsilon':eps,'gamma':gam,
            'mean_inner_MAE_days':float(np.mean(maes)),
            'sd_inner_MAE_days':float(np.std(maes,ddof=1))
        })

    combo_df=pd.DataFrame(combo_scores).sort_values(
        ['mean_inner_MAE_days','sd_inner_MAE_days','C','epsilon','gamma'],
        ascending=[True,True,True,True,True]
    )
    best=combo_df.iloc[0].to_dict()
    selected_rows.append({'outer_fold':outer,**best})

    tr=idx_for_eggs(outer_train); te=idx_for_eggs(outer_test)
    A,B=prepare_svr_matrices(tr,te)
    pred=fit_predict_svr_prepared(
        A,y[tr],B,best['C'],best['epsilon'],best['gamma']
    )
    for j,idx in enumerate(te):
        oof_rows.append({
            'sample':int(samples[idx]),'storage_days':float(y[idx]),
            'outer_fold':outer,'y_pred':float(pred[j]),
            'C':best['C'],'epsilon':best['epsilon'],'gamma':best['gamma']
        })

    # Persistent progress after each outer fold
    pd.DataFrame(search_rows).to_csv(search_path,index=False)
    pd.DataFrame(selected_rows).to_csv(selected_path,index=False)
    pd.DataFrame(oof_rows).to_csv(oof_path,index=False)
    print('PASS outer',outer,'best',best)

print('PASS — optimized SVR sensitivity outer evaluation completed.')

Hyperparameter combinations: 125
PASS outer 1 best {'C': 1000000.0, 'epsilon': 1.0, 'gamma': 3e-06, 'mean_inner_MAE_days': 2.2771617600852747, 'sd_inner_MAE_days': 0.1258072796958662}
PASS outer 2 best {'C': 1000000.0, 'epsilon': 2.0, 'gamma': 1e-06, 'mean_inner_MAE_days': 2.2820815969182604, 'sd_inner_MAE_days': 0.20470765993956963}
PASS outer 3 best {'C': 1000000.0, 'epsilon': 1.0, 'gamma': 1e-06, 'mean_inner_MAE_days': 2.245516358882647, 'sd_inner_MAE_days': 0.22763488784793837}
PASS outer 4 best {'C': 1000000.0, 'epsilon': 2.0, 'gamma': 3e-06, 'mean_inner_MAE_days': 2.2146893796957343, 'sd_inner_MAE_days': 0.16758396141097417}
PASS outer 5 best {'C': 100000.0, 'epsilon': 1.0, 'gamma': 1e-05, 'mean_inner_MAE_days': 2.258204252616105, 'sd_inner_MAE_days': 0.18209409737075322}
PASS — optimized SVR sensitivity outer evaluation completed.


In [9]:
# 7. Compare frozen primary SVR with sensitivity SVR
sens_oof=pd.DataFrame(oof_rows)
sens_metrics=metrics(sens_oof.storage_days,sens_oof.y_pred)

primary=pd.read_csv(NB03_DIR/'NB03_pooled_oof_metrics.csv')
primary_svr=primary[primary.model=='SVR'].iloc[0].to_dict()

comparison=pd.DataFrame([{
    'analysis':'Primary frozen NB03 SVR',
    'MAE_days':primary_svr.get('MAE_days',primary_svr.get('MAE')),
    'RMSE_days':primary_svr.get('RMSE_days',primary_svr.get('RMSE')),
    'R2':primary_svr.get('R2',primary_svr.get('R²'))
},{
    'analysis':'Reviewer-requested wider-grid sensitivity SVR',
    **{k:sens_metrics[k] for k in ['MAE_days','RMSE_days','R2']}
}])
comparison.to_csv(OUT_DIR/'NB09B_SVR_PRIMARY_VS_SENSITIVITY.csv',index=False)
display(comparison)

,analysis,MAE_days,RMSE_days,R2
0,Primary frozen NB03 SVR,2.194628,2.716171,0.816706
1,Reviewer-requested wider-grid sensitivity SVR,2.212318,2.735211,0.814127


## B. Operational storage-phase classification from frozen OOF predictions

This analysis does **not** claim microbiological freshness, safety, acceptability, or rejection.

It asks a narrower operational question: if the continuous storage-time estimate is converted into three **storage-phase categories**, how often is the correct phase recovered?

True phases:
- Early: 0–7 days
- Middle: 8–14 days
- Late: 15–21 days

Predicted continuous values are assigned using boundaries at 7.5 and 14.5 days. No clipping is required.

In [10]:
# 8. Load frozen valid egg-disjoint OOF predictions
oof03=pd.read_csv(NB03_DIR/'NB03_oof_predictions.csv')
oof03=oof03[oof03.model.isin(['PLSR','SVR'])][['sample','storage_days','outer_fold','model','y_pred']]

oof04=pd.read_csv(NB04_DIR/'NB04_oof_predictions_seedmean.csv')
# Normalize possible column variants
if 'y_pred_seedmean' in oof04.columns and 'y_pred' not in oof04.columns:
    oof04=oof04.rename(columns={'y_pred_seedmean':'y_pred'})
oof04=oof04[oof04.model.isin(['ANN','SimpleRNN','LSTM','BiLSTM'])][
    ['sample','storage_days','outer_fold','model','y_pred']
]

oof=pd.concat([oof03,oof04],ignore_index=True)
assert oof.groupby('model').size().eq(660).all()
print(oof.groupby('model').size())

model
ANN          660
BiLSTM       660
LSTM         660
PLSR         660
SVR          660
SimpleRNN    660
dtype: int64


In [11]:
# 9. Phase classification metrics
from sklearn.metrics import (
    confusion_matrix, classification_report, balanced_accuracy_score,
    f1_score, cohen_kappa_score, accuracy_score, recall_score,
    precision_score
)

LABELS=['Early (0–7)','Middle (8–14)','Late (15–21)']

def true_phase(day):
    if day <= 7: return LABELS[0]
    if day <= 14: return LABELS[1]
    return LABELS[2]

def predicted_phase(pred):
    if pred < 7.5: return LABELS[0]
    if pred < 14.5: return LABELS[1]
    return LABELS[2]

summary=[]
per_class=[]
cm_long=[]

for model,g in oof.groupby('model'):
    yt=np.array([true_phase(v) for v in g.storage_days])
    yp=np.array([predicted_phase(v) for v in g.y_pred])

    cm=confusion_matrix(yt,yp,labels=LABELS)
    summary.append({
        'model':model,
        'accuracy':accuracy_score(yt,yp),
        'balanced_accuracy':balanced_accuracy_score(yt,yp),
        'macro_F1':f1_score(yt,yp,labels=LABELS,average='macro'),
        'weighted_F1':f1_score(yt,yp,labels=LABELS,average='weighted'),
        'cohen_kappa':cohen_kappa_score(yt,yp,labels=LABELS),
    })

    for i,label in enumerate(LABELS):
        tp=cm[i,i]
        fn=cm[i,:].sum()-tp
        fp=cm[:,i].sum()-tp
        tn=cm.sum()-tp-fn-fp
        sensitivity=tp/(tp+fn) if tp+fn else np.nan
        specificity=tn/(tn+fp) if tn+fp else np.nan
        precision=tp/(tp+fp) if tp+fp else np.nan
        f1=2*precision*sensitivity/(precision+sensitivity) if precision+sensitivity else np.nan
        per_class.append({
            'model':model,'storage_phase':label,
            'sensitivity_recall':sensitivity,
            'specificity':specificity,
            'precision':precision,
            'F1':f1,
            'n_true':int(cm[i,:].sum())
        })

    for i,a in enumerate(LABELS):
        for j,b in enumerate(LABELS):
            cm_long.append({'model':model,'true_phase':a,'predicted_phase':b,'count':int(cm[i,j])})

phase_summary=pd.DataFrame(summary).sort_values('balanced_accuracy',ascending=False)
phase_class=pd.DataFrame(per_class)
phase_cm=pd.DataFrame(cm_long)

phase_summary.to_csv(OUT_DIR/'NB09B_storage_phase_classification_summary.csv',index=False)
phase_class.to_csv(OUT_DIR/'NB09B_storage_phase_per_class_metrics.csv',index=False)
phase_cm.to_csv(OUT_DIR/'NB09B_storage_phase_confusion_matrices_long.csv',index=False)

display(phase_summary)
display(phase_class[phase_class.model.isin(['SVR','PLSR','ANN'])])

,model,accuracy,balanced_accuracy,macro_F1,weighted_F1,cohen_kappa
4,SVR,0.786364,0.785913,0.788418,0.791776,0.680107
3,PLSR,0.768182,0.766071,0.769184,0.773615,0.652632
0,ANN,0.748485,0.744444,0.747509,0.752381,0.622415
1,BiLSTM,0.468182,0.467063,0.449789,0.454644,0.205228
2,LSTM,0.448485,0.452778,0.397469,0.401487,0.181688
5,SimpleRNN,0.375758,0.382341,0.278892,0.282014,0.076233


,model,storage_phase,sensitivity_recall,specificity,precision,F1,n_true
0,ANN,Early (0–7),0.833333,0.933333,0.877193,0.854701,240
1,ANN,Middle (8–14),0.695238,0.777778,0.593496,0.640351,210
2,ANN,Late (15–21),0.704762,0.915556,0.795699,0.747475,210
9,PLSR,Early (0–7),0.812500,0.964286,0.928571,0.866667,240
10,PLSR,Middle (8–14),0.742857,0.782222,0.614173,0.672414,210
11,PLSR,Late (15–21),0.742857,0.911111,0.795918,0.768473,210
12,SVR,Early (0–7),0.795833,0.971429,0.940887,0.862302,240
13,SVR,Middle (8–14),0.776190,0.791111,0.634241,0.698073,210
14,SVR,Late (15–21),0.785714,0.922222,0.825000,0.804878,210


### Interpretation guardrail

Use the terms **early/middle/late storage phase**, not “fresh / acceptable / reject”, unless independent biological or regulatory thresholds are available. Storage duration is not identical to freshness or food safety.

## C. Quantify attenuation / regression toward the center

For each competitive model, fit:

\[
\widehat{y} = \alpha + \beta y
\]

where \(y\) is observed storage day and \(\widehat{y}\) is the frozen OOF prediction.

A slope \(\beta < 1\) quantifies compression of predictions toward the center of the experimental range. This is descriptive and does not establish a physicochemical mechanism by itself.

In [12]:
# 10. Predicted-on-observed regression slopes with egg-level bootstrap CI
from scipy.stats import linregress

def slope_intercept(x,z):
    r=linregress(np.asarray(x,float),np.asarray(z,float))
    return r.slope,r.intercept,r.rvalue**2

rng=np.random.default_rng(62026)
slope_rows=[]

for model in ['SVR','PLSR','ANN']:
    g=oof[oof.model==model].copy()
    slope,intercept,r2line=slope_intercept(g.storage_days,g.y_pred)

    eggs=sorted(g['sample'].unique())
    boots=[]
    for b in range(10000):
        chosen=rng.choice(eggs,size=len(eggs),replace=True)
        chunks=[]
        for new_id,egg in enumerate(chosen):
            q=g[g['sample']==egg].copy()
            q['_boot_cluster']=new_id
            chunks.append(q)
        gb=pd.concat(chunks,ignore_index=True)
        boots.append(slope_intercept(gb.storage_days,gb.y_pred)[0])

    lo,hi=np.quantile(boots,[0.025,0.975])
    slope_rows.append({
        'model':model,
        'predicted_on_observed_slope':slope,
        'intercept_days':intercept,
        'line_R2':r2line,
        'slope_bootstrap95_low':lo,
        'slope_bootstrap95_high':hi,
        'bootstrap_resamples':10000,
        'bootstrap_unit':'egg'
    })

slopes=pd.DataFrame(slope_rows)
slopes.to_csv(OUT_DIR/'NB09B_predicted_on_observed_attenuation_slopes.csv',index=False)
display(slopes)

,model,predicted_on_observed_slope,intercept_days,line_R2,slope_bootstrap95_low,slope_bootstrap95_high,bootstrap_resamples,bootstrap_unit
0,SVR,0.854599,1.674598,0.818813,0.824557,0.884720,10000,egg
1,PLSR,0.845429,1.629185,0.798959,0.819710,0.871395,10000,egg
2,ANN,0.822630,1.703682,0.782881,0.787722,0.860033,10000,egg


In [13]:
# 11. Additional reviewer-facing notes / exact wording anchors
notes = {
    'scope_sentence': (
        'The reported generalization estimates apply to previously unseen eggs from the same '
        'acquisition campaign and do not establish transferability across farms, batches, '
        'instruments, breeds, seasons, temperatures, or humidity regimes.'
    ),
    'clipping_sentence': (
        'Clipping predictions to the experimental interval [0, 21] days was evaluated only as a '
        'secondary sensitivity analysis because such bounds use prior knowledge of the experimental '
        'support; all primary generalization metrics remain unclipped.'
    ),
    'multiplicity_sentence': (
        'Holm correction was applied within each prespecified family of pairwise tests; no additional '
        'global multiplicity correction was imposed across the distinct benchmark and order-ablation '
        'families, which address different inferential questions.'
    ),
    'perfect_separation_sentence': (
        'For the nine competitive-versus-recurrent comparisons, all 30 egg-level paired differences '
        'favored the competitive model, producing rank-biserial |r| = 1 and identical minimum '
        'Wilcoxon p-values; this represents complete directional separation rather than fine-grained '
        'evidence about the magnitude of differences among recurrent architectures.'
    ),
    'balanced_design_sentence': (
        'Because the experimental target distribution is perfectly balanced across days 0–21 for '
        'every egg, pooled R² and the DummyMean benchmark reflect this uniform experimental support '
        'and may differ from values obtained under an operationally imbalanced storage-time distribution.'
    ),
}
(OUT_DIR/'NB09B_manuscript_wording_anchors.json').write_text(json.dumps(notes,indent=2),encoding='utf-8')
print(json.dumps(notes,indent=2))

{
  "scope_sentence": "The reported generalization estimates apply to previously unseen eggs from the same acquisition campaign and do not establish transferability across farms, batches, instruments, breeds, seasons, temperatures, or humidity regimes.",
  "clipping_sentence": "Clipping predictions to the experimental interval [0, 21] days was evaluated only as a secondary sensitivity analysis because such bounds use prior knowledge of the experimental support; all primary generalization metrics remain unclipped.",
  "multiplicity_sentence": "Holm correction was applied within each prespecified family of pairwise tests; no additional global multiplicity correction was imposed across the distinct benchmark and order-ablation families, which address different inferential questions.",
  "perfect_separation_sentence": "For the nine competitive-versus-recurrent comparisons, all 30 egg-level paired differences favored the competitive model, producing rank-biserial |r| = 1 and identical minim

In [14]:
# 12. Run summary
run_summary={
    'run_revision':RUN_REVISION,
    'role':'review-driven secondary analyses; primary NB01-NB08 unchanged',
    'svr_sensitivity_is_posthoc':True,
    'storage_phase_classes_are_freshness_classes':False,
    'storage_phase_boundaries_days':{'early':[0,7],'middle':[8,14],'late':[15,21]},
    'attenuation_bootstrap_unit':'egg',
    'completed':True
}
(OUT_DIR/'NB09B_run_summary.json').write_text(json.dumps(run_summary,indent=2),encoding='utf-8')
print('NB09B COMPLETED')

NB09B COMPLETED


In [15]:
# ============================================================
# DESCARGAR RESULTADOS NB09B EN ZIP
# ============================================================

from pathlib import Path
import shutil
from google.colab import files

RESULTS_DIR = Path(
    "/content/drive/MyDrive/NIR_HUEVOS_PAPER_REBUILD_2026/"
    "05_RESULTS/NB09B_REVIEWER_ADDITIONAL_ANALYSES"
)

ZIP_BASE = Path("/content/NB09B_RESULTS_REVIEWER_ADDITIONAL_ANALYSES")

assert RESULTS_DIR.exists(), f"No existe la carpeta: {RESULTS_DIR}"

zip_path = shutil.make_archive(
    str(ZIP_BASE),
    "zip",
    root_dir=str(RESULTS_DIR)
)

print("✅ ZIP creado correctamente:")
print(zip_path)

files.download(zip_path)

✅ ZIP creado correctamente:
/content/NB09B_RESULTS_REVIEWER_ADDITIONAL_ANALYSES.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>